# ONT ORFanage Spidroin Full-Length Screening

Annotate CDS features for ONT isoforms with ORFanage, translate proteins with `gffread`, and screen high-confidence full-length spidroins with protein-level HMMER terminal-domain models.

Intermediate files are written to `data/interim/{TASK_NAME}/`; final tables and FASTA files are written to `data/processed/{TASK_NAME}/`.

## Configuration

Import packages and define run-specific paths. `TASK_NAME` includes a timestamp suffix so each run is isolated.

In [1]:
from datetime import datetime

import polars as pl

from spider_silkome_module import (
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROJ_ROOT,
    RAW_DATA_DIR,
    run_cmd,
)

2026-05-26 18:13:31.063 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome


In [14]:
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
TASK_NAME = f"ont_orfanage_spidroin_{RUN_TIMESTAMP}"

isoform_dir = PROJ_ROOT / "workflow" / "ONT_RNA_align" / "results" / "isoforms"
genome_dir = RAW_DATA_DIR / "spider_genome"
hmm_dir = INTERIM_DATA_DIR / "spider_silkome_20251222" / "hmmbuild_output"

interim_dir = INTERIM_DATA_DIR / TASK_NAME
processed_dir = PROCESSED_DATA_DIR / TASK_NAME

THREADS = 70
EVALUE = 1e-10
MIN_HMM_COVERAGE = 0.90
TERMINAL_WINDOW = 300
FORCE = False

SMOKE_SPECIES = "Araneus_ventricosus"
RUN_SMOKE_TEST = False

interim_dir, processed_dir

(PosixPath('/home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260526_181751'),
 PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260526_181751'))

## Step 1: Preflight

Check that required executables and input directories are available.

In [15]:
preflight_tsv = interim_dir / "preflight.tsv"

run_cmd(
    f"pixi run python -m spider_silkome_module.ont_orfanage_spidroin preflight \
        --isoform-dir {isoform_dir} \
        --genome-dir {genome_dir} \
        --hmm-dir {hmm_dir} \
        --output {preflight_tsv}",
    outputs=[preflight_tsv],
    force=FORCE,
)

pl.read_csv(preflight_tsv, separator="\t")

2026-05-26 18:17:54.450 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome


2026-05-26 18:17:54.714 | INFO     | __main__:preflight:928 - executable:orfanage: True /home/gyk/project/spider_silkome/.pixi/envs/default/bin/orfanage
2026-05-26 18:17:54.715 | INFO     | __main__:preflight:928 - executable:gffread: True /home/gyk/project/spider_silkome/.pixi/envs/default/bin/gffread
2026-05-26 18:17:54.716 | INFO     | __main__:preflight:928 - executable:hmmsearch: True /home/gyk/project/spider_silkome/.pixi/envs/default/bin/hmmsearch
2026-05-26 18:17:54.716 | INFO     | __main__:preflight:928 - isoform_dir_exists: True /home/gyk/project/spider_silkome/workflow/ONT_RNA_align/results/isoforms
2026-05-26 18:17:54.716 | INFO     | __main__:preflight:928 - genome_dir_exists: True /home/gyk/project/spider_silkome/data/raw/spider_genome
2026-05-26 18:17:54.716 | INFO     | __main__:preflight:928 - hmm_dir_exists: True /home/gyk/project/spider_silkome/data/interim/spider_silkome_20251222/hmmbuild_output
2026-05-26 18:17:54.716 | INFO     | __main__:preflight:928 - isoform_

check,status,value
str,bool,str
"""executable:orfanage""",true,"""/home/gyk/project/spider_silko…"
"""executable:gffread""",true,"""/home/gyk/project/spider_silko…"
"""executable:hmmsearch""",true,"""/home/gyk/project/spider_silko…"
"""isoform_dir_exists""",true,"""/home/gyk/project/spider_silko…"
"""genome_dir_exists""",true,"""/home/gyk/project/spider_silko…"
"""hmm_dir_exists""",true,"""/home/gyk/project/spider_silko…"
"""isoform_gtf_count""",true,"""1"""
"""hmm_count""",true,"""29"""


## Step 2: Discover Species

Build a manifest that pairs ONT isoform GTF files with matching genome FASTA and template GFF annotations.

In [16]:
manifest_tsv = interim_dir / "species_manifest.tsv"

run_cmd(
    f"pixi run python -m spider_silkome_module.ont_orfanage_spidroin discover \
        --isoform-dir {isoform_dir} \
        --genome-dir {genome_dir} \
        --output {manifest_tsv}",
    outputs=[manifest_tsv],
    force=FORCE,
)

manifest = pl.read_csv(manifest_tsv, separator="\t")
manifest

2026-05-26 18:17:58.860 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome


2026-05-26 18:17:59.128 | SUCCESS  | __main__:discover:944 - Wrote /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260526_181751/species_manifest.tsv with 1/1 matched species


species,isoform_gtf,genome_fasta,template_gff,status,reason
str,str,str,str,str,str
"""Araneus_ventricosus""","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""/home/gyk/project/spider_silko…","""matched""",null


## Optional Smoke Test

Run the complete workflow for one species before starting the full batch by setting `RUN_SMOKE_TEST = True` in the configuration cell.

In [17]:
if RUN_SMOKE_TEST:
    smoke_interim_dir = interim_dir / "smoke_test"
    smoke_processed_dir = processed_dir / "smoke_test"
    smoke_target_tsv = smoke_processed_dir / "target_classification.tsv"

    run_cmd(
        f"pixi run python -m spider_silkome_module.ont_orfanage_spidroin run \
            --isoform-dir {isoform_dir} \
            --genome-dir {genome_dir} \
            --hmm-dir {hmm_dir} \
            --interim-dir {smoke_interim_dir} \
            --processed-dir {smoke_processed_dir} \
            --species {SMOKE_SPECIES} \
            --threads {THREADS} \
            --evalue {EVALUE} \
            --min-hmm-coverage {MIN_HMM_COVERAGE} \
            --terminal-window {TERMINAL_WINDOW}",
        outputs=[smoke_target_tsv],
        force=FORCE,
    )

    display(pl.read_csv(smoke_processed_dir / "orfanage_protein_qc.tsv", separator="\t"))
    display(pl.read_csv(smoke_target_tsv, separator="\t").head())

## Step 3: Run Full Workflow

Run ORFanage, translate ORFanage CDS, search HMMER terminal-domain profiles, and write final processed outputs.

In [18]:
target_classification_tsv = processed_dir / "target_classification.tsv"

run_cmd(
    f"pixi run python -m spider_silkome_module.ont_orfanage_spidroin run \
        --isoform-dir {isoform_dir} \
        --genome-dir {genome_dir} \
        --hmm-dir {hmm_dir} \
        --interim-dir {interim_dir} \
        --processed-dir {processed_dir} \
        --threads {THREADS} \
        --evalue {EVALUE} \
        --min-hmm-coverage {MIN_HMM_COVERAGE} \
        --terminal-window {TERMINAL_WINDOW}",
    outputs=[target_classification_tsv],
    force=True,
)

target_classification_tsv

2026-05-26 18:18:06.301 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome
Species:   0%|          | 0/1 [00:04<?, ?it/s]loading reference genome
loading reference transcriptomes


2026-05-26 18:18:10.631 | INFO     | __main__:run_command:222 - orfanage --reference /home/gyk/project/spider_silkome/data/raw/spider_genome/064.Araneus_ventricosus/Araneus_ventricosus.fa --query /home/gyk/project/spider_silkome/workflow/ONT_RNA_align/results/isoforms/Araneus_ventricosus.isoforms.gtf --output /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260526_181751/orfanage/Araneus_ventricosus.orfanage.gtf --stats /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260526_181751/orfanage/Araneus_ventricosus.stats.tsv --threads 70 --use_id --cleanq --cleant --rescue /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260526_181751/template_gff_cleaned/Araneus_ventricosus.cleaned.gff


sorting reference transcriptome
start removing duplicates
loading query transcriptome
bundling transcriptome
starting main evaluation
Done
Species:   0%|          | 0/1 [00:21<?, ?it/s]

2026-05-26 18:18:27.909 | INFO     | __main__:run_command:222 - gffread /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260526_181751/orfanage/Araneus_ventricosus.orfanage.gtf -g /home/gyk/project/spider_silkome/data/raw/spider_genome/064.Araneus_ventricosus/Araneus_ventricosus.fa -y /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260526_181751/proteins/Araneus_ventricosus.orfanage.proteins.faa


Species:   0%|          | 0/1 [00:27<?, ?it/s]

2026-05-26 18:18:33.592 | INFO     | __main__:run:1020 - Araneus_ventricosus: 20216 translated proteins
2026-05-26 18:18:33.592 | INFO     | __main__:run_command:222 - hmmsearch --cpu 70 --noali -E 1e-10 --domE 1e-10 --incE 1e-10 --incdomE 1e-10 -o /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260526_181751/hmmer/Araneus_ventricosus.hmmsearch.txt --tblout /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260526_181751/hmmer/Araneus_ventricosus.tblout --domtblout /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260526_181751/hmmer/Araneus_ventricosus.domtblout /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260526_181751/hmmer/spidroin_terminals.all.hmm /home/gyk/project/spider_silkome/data/interim/ont_orfanage_spidroin_20260526_181751/proteins/Araneus_ventricosus.orfanage.proteins.normalized.faa


Species: 100%|██████████| 1/1 [00:35<00:00, 35.71s/it]


2026-05-26 18:18:43.818 | SUCCESS  | __main__:run_classification:876 - Classified 9 targets: 0 full-length, 9 partial/non-full; wrote 0 FASTA records.


PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260526_181751/target_classification.tsv')

## Results

Inspect final full-length candidates, pivot counts, and validation checks.

In [19]:
target_classification = pl.read_csv(processed_dir / "target_classification.tsv", separator="\t")
full_length = pl.read_csv(processed_dir / "full_length_spidroins.tsv", separator="\t")
partial = pl.read_csv(processed_dir / "partial_spidroins.tsv", separator="\t")
pivot = pl.read_csv(processed_dir / "spidroin_pivot_table.tsv", separator="\t")
validation = pl.read_csv(processed_dir / "validation_summary.tsv", separator="\t")

print(f"target classifications: {target_classification.shape}")
print(f"full-length candidates: {full_length.shape}")
print(f"partial/non-full candidates: {partial.shape}")

display(validation)
display(pivot)
full_length.head()

target classifications: (9, 38)
full-length candidates: (0, 38)
partial/non-full candidates: (9, 38)


check,status,value
str,bool,str
"""full_length_fasta_count_matche…",true,"""0/0"""
"""full_length_fasta_duplicate_id…",true,"""0"""
"""full_length_same_type_ntd_ctd""",true,"""0"""


species
str
"""Araneus_ventricosus"""


species,target_id,tlen,classification,spidroin_type,ntd_profile,ctd_profile,ntd_spidroin_type,ctd_spidroin_type,ntd_end_ok,ctd_end_ok,ntd_c_evalue,ntd_dom_score,ntd_env_from,ntd_env_to,ntd_hmm_from,ntd_hmm_to,ntd_hmm_coverage,ntd_env_coverage,ctd_c_evalue,ctd_dom_score,ctd_env_from,ctd_env_to,ctd_hmm_from,ctd_hmm_to,ctd_hmm_coverage,ctd_env_coverage,num_ntd_hits,num_ctd_hits,original_transcript_id,gene_id,scaffold,strand,start,end,orfanage_status,orfanage_template,summed_dom_score
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str


In [20]:
processed_outputs = {
    "target_classification": processed_dir / "target_classification.tsv",
    "full_length_spidroins": processed_dir / "full_length_spidroins.tsv",
    "partial_spidroins": processed_dir / "partial_spidroins.tsv",
    "spidroin_pivot_table": processed_dir / "spidroin_pivot_table.tsv",
    "full_length_fasta": processed_dir / "full_length_spidroins.fasta",
    "by_type_dir": processed_dir / "by_type",
}

processed_outputs

{'target_classification': PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260526_181751/target_classification.tsv'),
 'full_length_spidroins': PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260526_181751/full_length_spidroins.tsv'),
 'partial_spidroins': PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260526_181751/partial_spidroins.tsv'),
 'spidroin_pivot_table': PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260526_181751/spidroin_pivot_table.tsv'),
 'full_length_fasta': PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260526_181751/full_length_spidroins.fasta'),
 'by_type_dir': PosixPath('/home/gyk/project/spider_silkome/data/processed/ont_orfanage_spidroin_20260526_181751/by_type')}